# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeerMusabih/FlyRank-AI-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Note: Please ignore the first code cell. it is only to ready the notebook to connect to the github repository.

In [11]:
!git clone https://github.com/MeerMusabih/FlyRank-AI-Internship.git

%cd FlyRank-AI-Internship

import os
import pandas as pd

print("Current directory:")
print(os.getcwd())

dataset_path = "data/raw/content_refresh_anonymized.csv"

print("\nDataset exists:", os.path.exists(dataset_path))

if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)

    print("\nDataset loaded successfully!")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    display(df.head())
else:
    print("\nDataset NOT found.")
    print("\nFiles in data/raw:")
    !ls -lah data/raw

Cloning into 'FlyRank-AI-Internship'...
remote: Enumerating objects: 182, done.
remote: Counting objects: 100% (182/182), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 182 (delta 79), reused 86 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (182/182), 2.16 MiB | 16.52 MiB/s, done.
Resolving deltas: 100% (79/79), done.
/content/FlyRank-AI-Internship/FlyRank-AI-Internship/FlyRank-AI-Internship/FlyRank-AI-Internship/FlyRank-AI-Internship
Current directory:
/content/FlyRank-AI-Internship/FlyRank-AI-Internship/FlyRank-AI-Internship/FlyRank-AI-Internship/FlyRank-AI-Internship

Dataset exists: True

Dataset loaded successfully!
Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. Two paper findings + my methodology questions

### Finding 1 — The Anatomy of Growing Content

The paper reports that growing pages were longer and younger than declining pages: growing content averaged about 3.2K words and 184 days old, while declining content averaged about 2.3K words and 230 days old.

The label comes from `trend_direction`, which is calculated from the change in impressions between the most recent 30 days and the previous 30 days. Pages with more than 10% growth are labeled `up`, while pages with more than 10% decline are labeled `down`.

My methodology question is whether this validation design is strong enough to support the broader interpretation. The comparison is useful as an observed portfolio-level difference, but it is observational. Age and word count may be associated with growth without causing it. The paper itself treats this finding as directional rather than causal.

### Finding 2 — The Content Performance Curve

The paper reports that content reaches its strongest health score around 61–90 days and that health declines substantially in the 271–365 day range. It also reports a rebound among 365+ day content that has been refreshed.

The label or grouping here comes from content age and freshness buckets rather than a supervised outcome label. Age is measured as days since content creation, while freshness is days since the last update.

My methodology question is whether the age and freshness comparisons can distinguish the effect of refreshing content from other differences between pages. Older pages that were refreshed may already have had stronger demand or received more attention, so the observed improvement should not be interpreted as proof that refreshing alone caused the increase. The finding is useful for prioritization, but it is best treated as directional decision-support.

## 2. My model under an honest split (before/after)

### Honest split

The Week-5 model was originally evaluated using a random split. For this audit, I re-evaluate the model using a grouped split so that pages from the same client do not appear in both the training and test sets.

This gives a more conservative estimate of how well the model may generalize to unseen clients.
### Before: Week-5 grouped validation

The Week-5 Random Forest was evaluated using a client-grouped 80/20 split. The model achieved Precision@20 of 0.70 and Precision@50 of 0.68.

This is a stronger validation design than a simple random row-level split because pages from the same client were kept together. However, it does not test whether the model generalizes forward in time.

In [6]:
!git clone https://github.com/MeerMusabih/FlyRank-AI-Internship.git
%cd FlyRank-AI-Internship

Cloning into 'FlyRank-AI-Internship'...
remote: Enumerating objects: 182, done.
remote: Counting objects: 100% (182/182), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 182 (delta 79), reused 86 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (182/182), 2.16 MiB | 11.59 MiB/s, done.
Resolving deltas: 100% (79/79), done.
/content/FlyRank-AI-Internship/FlyRank-AI-Internship


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.